# Local Hybrid Chat Debug

Notebook นี้ไว้ถาม chatbot แบบ local แล้วดูว่าแต่ละคำตอบมาจากอะไร เช่น `fast_path`, `rule`, `RAG`, `vector` หรือ `Local LLM` พร้อมเวลาที่ใช้

เพิ่มในเวอร์ชันนี้:
- มี `SESSION_ID` / `SECTION_ID` ต่อ 1 รอบการรัน notebook
- `ask(...)` จะจำบริบทใน session เดียวกัน เช่น ถามชื่อเกมก่อน แล้วถามต่อว่า “ปุ่มทั้งหมดมีอะไรบ้าง”
- เปิดทดลอง Facts-only LLM Composer ได้ด้วย `ask(..., use_facts_composer=True)` หรือ `ask_with_composer(...)`
- log ทุกคำถามลงไฟล์ JSONL อัตโนมัติ แยกตาม session
- `/exit` ใน `chat_loop()` จะ save log แล้วออกจาก loop

ก่อนใช้ Local LLM ให้เปิด Ollama ไว้ก่อน และเช็กว่ามี model ที่ตั้งไว้ในเครื่อง เช่น `qwen3:4b` หรือ `qwen2.5:3b`


In [ ]:
from __future__ import annotations

import json
import os
import sys
import time
import urllib.request
from dataclasses import asdict, is_dataclass
from datetime import datetime
from pathlib import Path
from uuid import uuid4

PROJECT_ROOT = Path(r"C:\Users\Chokhun\Downloads\Learn-LLM\18_PSU_Esports_Update_Route_Data")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# เปลี่ยน model ได้ตรงนี้ ถ้าอยากลอง Qwen3 ให้เปลี่ยนเป็น qwen3:4b แต่ qwen2.5:3b เหมาะกับแชทธรรมดากว่า
MODEL = os.getenv("PSU_CHATBOT_OLLAMA_MODEL", "qwen2.5:3b")
OLLAMA_URL = os.getenv("OLLAMA_URL", "http://127.0.0.1:11434")
GENERAL_TIMEOUT_SEC = "40"
GENERAL_NUM_PREDICT = "1024"
FACTS_COMPOSER_DEFAULT = os.getenv("PSU_FACTS_LLM_COMPOSER", "0")
FACTS_COMPOSER_TIMEOUT_SEC = os.getenv("PSU_FACTS_LLM_TIMEOUT_SEC", "8")
FACTS_COMPOSER_NUM_PREDICT = os.getenv("PSU_FACTS_LLM_NUM_PREDICT", "360")
TOOL_ROUTER_DEFAULT = os.getenv("PSU_LLM_TOOL_ROUTER", "0")
TOOL_ROUTER_TIMEOUT_SEC = os.getenv("PSU_TOOL_ROUTER_TIMEOUT_SEC", "1.2")
TOOL_ROUTER_NUM_PREDICT = os.getenv("PSU_TOOL_ROUTER_NUM_PREDICT", "160")

# เปิด fallback ให้ Local LLM มีสิทธิ์ตอบคำถาม general/out-of-domain
os.environ["PSU_EXPERIMENTAL_RAG_FALLBACK"] = "1"
os.environ["PSU_EXPERIMENTAL_ALLOW_LLM"] = "1"
os.environ["PSU_CHATBOT_OLLAMA_MODEL"] = MODEL
os.environ["OLLAMA_URL"] = OLLAMA_URL
os.environ["PSU_GENERAL_LLM_TIMEOUT_SEC"] = GENERAL_TIMEOUT_SEC
os.environ["PSU_GENERAL_LLM_NUM_PREDICT"] = GENERAL_NUM_PREDICT
os.environ["PSU_FACTS_LLM_COMPOSER"] = FACTS_COMPOSER_DEFAULT
os.environ["PSU_FACTS_LLM_TIMEOUT_SEC"] = FACTS_COMPOSER_TIMEOUT_SEC
os.environ["PSU_FACTS_LLM_NUM_PREDICT"] = FACTS_COMPOSER_NUM_PREDICT
os.environ["PSU_LLM_TOOL_ROUTER"] = TOOL_ROUTER_DEFAULT
os.environ["PSU_TOOL_ROUTER_TIMEOUT_SEC"] = TOOL_ROUTER_TIMEOUT_SEC
os.environ["PSU_TOOL_ROUTER_NUM_PREDICT"] = TOOL_ROUTER_NUM_PREDICT
# ปิด thinking สำหรับ Qwen3 เพื่อกัน token หมดก่อนมี final answer
os.environ["PSU_OLLAMA_THINK"] = "false"

from app.runtime.pipeline_answer import answer_question_pipeline_debug
from app.session.context_resolver import resolve_question_with_context
from app.session.chat_logger import write_chat_log

# ถ้าเคย import module ไปแล้วใน kernel เดิม ให้บังคับอัปเดต runtime config ด้วย
import app.pipeline.experimental_fallback as experimental_fallback
experimental_fallback.DEFAULT_MODEL = MODEL
experimental_fallback.DEFAULT_TIMEOUT_SEC = float(GENERAL_TIMEOUT_SEC)
experimental_fallback.DEFAULT_GENERAL_NUM_PREDICT = int(GENERAL_NUM_PREDICT)

SESSION_ID = f"local-ipynb-{datetime.now().strftime('%Y%m%d-%H%M%S')}-{uuid4().hex[:8]}"
SECTION_ID = SESSION_ID  # alias ตามที่คุยกัน ใช้แทนกันได้
LOG_DIR = PROJECT_ROOT / "reports" / "local_hybrid_chat_debug"
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = LOG_DIR / f"{SESSION_ID}.jsonl"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("MODEL =", MODEL)
print("OLLAMA_URL =", OLLAMA_URL)
print("GENERAL_TIMEOUT_SEC =", GENERAL_TIMEOUT_SEC)
print("SESSION_ID =", SESSION_ID)
print("FACTS_COMPOSER_DEFAULT =", FACTS_COMPOSER_DEFAULT)
print("TOOL_ROUTER_DEFAULT =", TOOL_ROUTER_DEFAULT)
print("LOG_PATH =", LOG_PATH)


def enable_tool_router(model: str | None = None, timeout_sec: float = 1.2, num_predict: int = 160):
    """เปิด LLM Tool Router สำหรับทดลองให้ model ช่วยเลือก route/tool ก่อนตอบ"""
    os.environ["PSU_LLM_TOOL_ROUTER"] = "1"
    os.environ["PSU_TOOL_ROUTER_TIMEOUT_SEC"] = str(timeout_sec)
    os.environ["PSU_TOOL_ROUTER_NUM_PREDICT"] = str(num_predict)
    if model:
        os.environ["PSU_TOOL_ROUTER_MODEL"] = model
    print("PSU_LLM_TOOL_ROUTER =", os.environ["PSU_LLM_TOOL_ROUTER"])
    print("PSU_TOOL_ROUTER_MODEL =", os.environ.get("PSU_TOOL_ROUTER_MODEL", os.environ.get("PSU_CHATBOT_OLLAMA_MODEL", MODEL)))
    print("PSU_TOOL_ROUTER_TIMEOUT_SEC =", os.environ["PSU_TOOL_ROUTER_TIMEOUT_SEC"])
    print("PSU_TOOL_ROUTER_NUM_PREDICT =", os.environ["PSU_TOOL_ROUTER_NUM_PREDICT"])


def disable_tool_router():
    """ปิด LLM Tool Router กลับไปใช้ heuristic/structured/fast/rule/RAG ตามเดิม"""
    os.environ["PSU_LLM_TOOL_ROUTER"] = "0"
    print("PSU_LLM_TOOL_ROUTER =", os.environ["PSU_LLM_TOOL_ROUTER"])


## ถ้าขึ้น `general_llm_unavailable`

- แปลว่า request ไปทาง Local LLM แล้ว แต่ Ollama/model ไม่ตอบกลับสำเร็จ
- ให้รัน cell เช็ก Ollama ด้านล่างก่อน ถ้าไม่ OK ให้เปิด `ollama serve` หรือเปลี่ยน `MODEL`
- ถ้าใช้ `qwen3:4b` แล้ว response ว่างหรือออกแนวข้อความคิด ให้ใช้ `qwen2.5:3b` ก่อน เพราะเหมาะกับแชทธรรมดากว่า
- หลังเปลี่ยน `MODEL`, `TIMEOUT`, หรือ `NUM_PREDICT` ให้ rerun cell ตั้งค่าด้านบนใหม่


In [ ]:
def check_ollama(model: str = MODEL, timeout_sec: float = 8.0) -> tuple[bool, str, float]:
    """เช็กแบบเร็วว่า Ollama/model เรียกได้ไหม"""
    payload = json.dumps({
        "model": model,
        "prompt": "ตอบด้วยข้อความต่อไปนี้เท่านั้น: OK",
        "stream": False,
        "think": False,
        "options": {"num_predict": 64},
    }).encode("utf-8")
    request = urllib.request.Request(
        f"{OLLAMA_URL.rstrip('/')}/api/generate",
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    started = time.perf_counter()
    try:
        with urllib.request.urlopen(request, timeout=timeout_sec) as response:
            data = json.loads(response.read().decode("utf-8"))
        return True, str(data.get("response", "")).strip(), round(time.perf_counter() - started, 3)
    except Exception as exc:
        return False, f"{type(exc).__name__}: {exc}", round(time.perf_counter() - started, 3)

ok, text, elapsed = check_ollama()
print("ollama_ok =", ok)
print("elapsed =", elapsed, "sec")
print("response/error =", text)


In [ ]:
history: list[dict] = []
closed = False


def json_default(value):
    if is_dataclass(value):
        return asdict(value)
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(k): json_default(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_default(v) for v in value]
    return str(value)


def plain(value):
    return json_default(value)


def classify_mode(mode: str) -> str:
    m = (mode or "").lower()
    if "general_llm" in m:
        return "Local LLM (general)"
    if "rag_llm" in m or "experimental_rag_llm" in m:
        return "RAG + Local LLM"
    if "structured" in m:
        return "Structured facts"
    if "game_control_vector" in m or "vector" in m:
        return "Vector/RAG control data"
    if "rag_direct" in m or "curated" in m or "hybrid" in m:
        return "RAG/curated"
    if "fast" in m or "deterministic" in m or "rule" in m:
        return "Fast/Rule"
    if "no_answer" in m or "disabled" in m or "unavailable" in m:
        return "No answer / LLM unavailable"
    return "Other pipeline"


def short_sources(hits: list[dict], limit: int = 5) -> list[dict[str, str]]:
    sources: list[dict[str, str]] = []
    seen: set[tuple[str, str]] = set()
    for hit in hits or []:
        meta = hit.get("metadata", {}) if isinstance(hit, dict) else {}
        source_id = str(hit.get("id") or meta.get("title") or meta.get("source_id") or "source")
        url = str(meta.get("source_url") or meta.get("url") or "")
        key = (source_id, url)
        if key in seen:
            continue
        seen.add(key)
        sources.append({"id": source_id, "url": url})
        if len(sources) >= limit:
            break
    return sources


def append_jsonl(path: Path, row: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8", newline="\n") as file:
        file.write(json.dumps(row, ensure_ascii=False, default=json_default) + "\n")


def save_history(path: str | Path | None = None):
    if path is None:
        path = LOG_PATH
    path = Path(path)
    # LOG_PATH ถูก append ทุก ask อยู่แล้ว ฟังก์ชันนี้ไว้ snapshot history ซ้ำแบบอ่านง่าย
    snapshot_path = path.with_name(path.stem + "_snapshot.json")
    snapshot = {
        "session_id": SESSION_ID,
        "section_id": SECTION_ID,
        "saved_at": datetime.now().isoformat(timespec="seconds"),
        "log_path": str(LOG_PATH),
        "history": history,
    }
    snapshot_path.write_text(json.dumps(snapshot, ensure_ascii=False, indent=2, default=json_default), encoding="utf-8")
    print("saved snapshot", snapshot_path)
    print("jsonl log", LOG_PATH)
    return snapshot_path


def new_session():
    """เริ่ม session ใหม่ใน kernel เดิม หลังจากเคย /exit หรืออยากล้างบริบททั้งหมด"""
    global SESSION_ID, SECTION_ID, LOG_PATH, history, closed
    history.clear()
    closed = False
    SESSION_ID = f"local-ipynb-{datetime.now().strftime('%Y%m%d-%H%M%S')}-{uuid4().hex[:8]}"
    SECTION_ID = SESSION_ID
    LOG_PATH = LOG_DIR / f"{SESSION_ID}.jsonl"
    print("new SESSION_ID =", SESSION_ID)
    print("LOG_PATH =", LOG_PATH)
    return SESSION_ID


def _trace_first(result, stage: str):
    for item in getattr(result, "trace", []) or []:
        if getattr(item, "stage", "") == stage:
            return item
    return None


def ask(question: str, *, show_trace: bool = False, show_sources: bool = True, use_context: bool = True, use_facts_composer: bool | None = None, return_row: bool = False) -> dict | None:
    global closed
    if closed:
        print("session เดิม /exit ไปแล้ว กำลังเปิด session ใหม่ให้")
        new_session()

    composer_enabled = (FACTS_COMPOSER_DEFAULT == "1") if use_facts_composer is None else bool(use_facts_composer)
    previous_composer = os.environ.get("PSU_FACTS_LLM_COMPOSER", "0")
    os.environ["PSU_FACTS_LLM_COMPOSER"] = "1" if composer_enabled else "0"

    recent_history = history[-12:] if use_context else []
    resolved = resolve_question_with_context(question, recent_history)
    final_question = resolved.resolved_question

    started = time.perf_counter()
    try:
        result = answer_question_pipeline_debug(
            final_question,
            experimental_rag_fallback=True,
            experimental_allow_llm=True,
        )
    finally:
        os.environ["PSU_FACTS_LLM_COMPOSER"] = previous_composer
    wall_sec = round(time.perf_counter() - started, 3)
    route = f"{result.route.category}/{result.route.intent}"
    source_type = classify_mode(result.mode)
    sources = short_sources(result.hits)
    facts_trace = _trace_first(result, "facts_composer")
    structured_trace = _trace_first(result, "structured_tool")
    row = {
        "session_id": SESSION_ID,
        "section_id": SECTION_ID,
        "time": datetime.now().isoformat(timespec="seconds"),
        "question": question,
        "resolved_question": final_question,
        "used_context": resolved.used_context,
        "context_game": resolved.context_game,
        "context_domain": resolved.context_domain,
        "context_operation": resolved.context_operation,
        "context_topic": resolved.context_topic,
        "context_reason": resolved.reason,
        "universal_intent": plain(result.universal_intent),
        "decision_artifact": plain(result.decision_artifact),
        "answer": result.answer,
        "source_type": source_type,
        "mode": result.mode,
        "route": route,
        "confidence": result.confidence,
        "elapsed_sec": result.elapsed,
        "wall_sec": wall_sec,
        "sources": sources,
        "facts_composer": plain(facts_trace),
        "structured_evidence": plain(getattr(structured_trace, "metadata", None)) if structured_trace else None,
        "trace": result.trace,
        "validation": result.validation,
    }

    history.append({"role": "user", "text": question, "resolved_text": final_question})
    history.append({
        "role": "assistant",
        "text": result.answer,
        "mode": result.mode,
        "route": route,
        "route_category": result.route.category,
        "route_intent": result.route.intent,
        "universal_intent": plain(result.universal_intent),
        "resolved_text": final_question,
    })
    append_jsonl(LOG_PATH, row)
    try:
        write_chat_log({
            "channel": "local_ipynb",
            "client_session_id": SESSION_ID,
            "question": question,
            "resolved_question": final_question,
            "context_resolution": resolved.to_dict(),
            "answer": result.answer,
            "mode": result.mode,
            "route_category": result.route.category,
            "route_intent": result.route.intent,
            "universal_intent": plain(result.universal_intent),
            "decision_artifact": plain(result.decision_artifact),
            "confidence": result.confidence,
            "latency_sec": result.elapsed,
            "wall_sec": wall_sec,
            "sources": sources,
            "experimental": {"rag_fallback": True, "allow_llm": True, "model": MODEL, "facts_composer": composer_enabled},
        })
    except Exception as exc:
        row["chat_logger_error"] = f"{type(exc).__name__}: {exc}"

    print("=" * 80)
    print("SESSION_ID:", SESSION_ID)
    print("Q:", question)
    if resolved.used_context:
        print("resolved_question:", final_question)
        print("context_game:", resolved.context_game)
        print("context_domain:", resolved.context_domain)
        print("context_operation:", resolved.context_operation)
    print(f"source_type: {source_type}")
    print(f"mode: {result.mode}")
    print(f"route: {route}")
    if getattr(result, "universal_intent", None) is not None:
        ui = result.universal_intent
        print(f"universal_intent: {ui.domain}/{ui.operation} | method={ui.method} | confidence={ui.confidence}")
    selected_candidate = (result.decision_artifact or {}).get("selected_candidate")
    if selected_candidate:
        print(f"selected_candidate: {selected_candidate.get('capability_id')} | score={selected_candidate.get('score')} | action={selected_candidate.get('action')}")
    if facts_trace is not None:
        print(f"facts_composer: {facts_trace.decision} | {facts_trace.confidence} | {facts_trace.detail}")
    print(f"confidence: {result.confidence}")
    print(f"elapsed: {result.elapsed}s | wall: {wall_sec}s")
    print("log:", LOG_PATH)
    print("-" * 80)
    print(result.answer)
    if show_sources:
        print("-" * 80)
        print("sources:")
        for item in sources or [{"id": "(none)", "url": ""}]:
            print("-", f"{item['id']} | {item['url']}" if item.get("url") else item["id"])
    if show_trace:
        print("-" * 80)
        print("trace:")
        for trace in result.trace[-8:]:
            print(f"- {trace.stage} | {trace.decision} | {trace.confidence} | {trace.detail}")
    if return_row:
        return row
    return None


def ask_with_composer(question: str, *, show_trace: bool = False, show_sources: bool = True, use_context: bool = True, return_row: bool = False) -> dict | None:
    """ถามแบบเปิด Facts-only LLM Composer เพื่อให้ Local LLM ช่วยเรียบเรียงจาก evidence"""
    return ask(question, show_trace=show_trace, show_sources=show_sources, use_context=use_context, use_facts_composer=True, return_row=return_row)


def ask_full(question: str, *, use_context: bool = True, use_facts_composer: bool | None = None) -> None:
    """แสดงคำตอบเต็มแบบไม่คืน dict ท้าย cell เพื่อลดปัญหา Jupyter ย่อ output เป็น ..."""
    ask(question, show_trace=False, show_sources=True, use_context=use_context, use_facts_composer=use_facts_composer, return_row=False)


In [ ]:
def chat_loop(*, show_trace: bool = False, show_sources: bool = True, use_facts_composer: bool | None = None):
    """ใช้ใน notebook ได้ แต่ถ้า Jupyter input ดูค้าง ให้ใช้ start_widget_chat() แทน"""
    global closed
    print("SESSION_ID =", SESSION_ID)
    print("พิมพ์คำถามได้เรื่อย ๆ | /exit เพื่อ save แล้วออก | /history | /session | /save | /clear")
    print("หมายเหตุ: loop นี้ซ่อน trace เป็นค่า default เพื่อให้ prompt รอบถัดไปเห็นง่ายขึ้น")
    while True:
        try:
            question = input("\nคุณ> ").strip()
        except (EOFError, KeyboardInterrupt):
            save_history()
            closed = True
            print("\nออกจาก chat loop แล้ว")
            break
        if not question:
            continue
        command = question.lower()
        if command in {"/exit", "exit", "quit", "q"}:
            save_history()
            closed = True
            print("ออกจาก chat loop แล้ว ถ้าจะเริ่ม session ใหม่ให้รัน cell ตั้งค่าใหม่")
            break
        if command == "/history":
            print(f"history_count = {len(history)} | session_id={SESSION_ID}")
            continue
        if command == "/session":
            print("SESSION_ID =", SESSION_ID)
            print("SECTION_ID =", SECTION_ID)
            print("LOG_PATH =", LOG_PATH)
            continue
        if command == "/save":
            save_history()
            continue
        if command == "/clear":
            history.clear()
            print("ล้าง memory ใน session นี้แล้ว แต่ SESSION_ID ยังเหมือนเดิม")
            continue
        ask(question, show_trace=show_trace, show_sources=show_sources, use_facts_composer=use_facts_composer)
        print("\nพร้อมถามต่อ")


def start_widget_chat(*, show_trace: bool = False, show_sources: bool = True, use_facts_composer: bool | None = None):
    """ช่องแชทสำหรับ Jupyter/VS Code Notebook ที่ไม่ติดปัญหา input() ค้างหลังถามข้อแรก"""
    try:
        import ipywidgets as widgets
        from IPython.display import display
    except Exception as exc:
        print(f"ใช้ widget ไม่ได้: {type(exc).__name__}: {exc}")
        print("ให้ใช้ chat_loop() หรือ ask('คำถาม') แทน")
        return None

    text = widgets.Text(
        placeholder="พิมพ์คำถาม แล้วกด Enter",
        description="คุณ>",
        layout=widgets.Layout(width="100%"),
    )
    send = widgets.Button(description="ถาม", button_style="primary")
    clear = widgets.Button(description="ล้างหน้าจอ")
    out = widgets.Output(layout={"border": "1px solid #ddd", "padding": "8px"})

    def submit(_=None):
        global closed
        if closed:
            return
        question = text.value.strip()
        text.value = ""
        if not question:
            return
        command = question.lower()
        with out:
            if command in {"/exit", "exit", "quit", "q"}:
                save_history()
                closed = True
                text.disabled = True
                send.disabled = True
                print("ออกจาก session แล้ว ถ้าจะเริ่มใหม่ให้รัน cell ตั้งค่าใหม่")
                return
            if command == "/clear":
                history.clear()
                out.clear_output()
                print("ล้าง memory และหน้าจอแล้ว")
                return
            if command == "/history":
                print(f"history_count = {len(history)} | session_id={SESSION_ID}")
                print("-" * 80)
                return
            if command == "/session":
                print("SESSION_ID =", SESSION_ID)
                print("SECTION_ID =", SECTION_ID)
                print("LOG_PATH =", LOG_PATH)
                print("-" * 80)
                return
            ask(question, show_trace=show_trace, show_sources=show_sources, use_facts_composer=use_facts_composer)
            print("\nพร้อมถามต่อ")

    def clear_output(_=None):
        out.clear_output()

    text.on_submit(submit)
    send.on_click(submit)
    clear.on_click(clear_output)
    display(widgets.VBox([widgets.HBox([text, send, clear]), out]))
    return {"input": text, "send": send, "clear": clear, "output": out}

# รันอย่างใดอย่างหนึ่งเพื่อเริ่มถามยาว ๆ
# start_widget_chat()                         # แนะนำสำหรับ notebook
# start_widget_chat(use_facts_composer=True)  # ทดลองให้ Local LLM ช่วยเรียบเรียงจาก facts
# chat_loop()                                 # ใช้ input() แบบเดิม
# chat_loop(use_facts_composer=True)          # loop แบบเปิด composer


## Start Chat Loop

รัน cell ด้านล่างเพื่อเริ่มถามแบบต่อเนื่อง ใช้ `/exit` เพื่อ save log แล้วออกจาก session

In [ ]:
question = "แต่ละหมวดมีใครบ้าง"  # ตัวอย่างคำถาม

ask_full(question)

In [ ]:
ask("เกมใน PS5 มีอะไรมั่ง", show_trace=True, return_row=False)

In [ ]:
ask_with_composer("เกมใน PS5 มีอะไรมั่ง")

In [ ]:
ask_full("สมาชิก PSU Esport ทั้งหมดมีใครบ้าง")